In [1]:
## General imports
from multiprocessing import Manager, Pool, cpu_count
from functools import partial
## Local imports
import get_data_matrix as getMat
import mapLearningUtils as mlu
import train_MappingJM as trMap

In [3]:
## Configuration for anthropometrics
pathAnthropoSave = './all_params/anthropometrics'
path_save_calib  = pathAnthropoSave + '_CALIB.pkl'
path_save_assist = pathAnthropoSave + '_ASSIST.pkl'
path_expe_calib  = './data_CALIB/'
path_expe_assist = './data_ASSIST/'
subjList_calib   = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17']
subjList_assist  = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S09', 'S10', 'S12', 'S14']

params_calib  = {'pathExpe': path_expe_calib, 'listSubj': subjList_calib, 'savePath': path_save_calib}
params_assist = {'pathExpe': path_expe_assist, 'listSubj': subjList_assist, 'savePath': path_save_assist}

## Get and save anthropometrics
getMat.get_all_anthropo_data(params_calib)
getMat.get_all_anthropo_data(params_assist)

In [2]:
## Set configuration
# Data handling
src_data       = ['CALIB', 'ASSIST']
data_type_list = ['exoPositions', 'exoVelocities', 'humanPositions', 'humanVelocities', 'wristSensor']
list_vars      = ['q3', 'q4', 'qd3', 'qd4', 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz', 'l_a', 'l_fa',
                  'shoulder_elv', 'elbow_flexion', 'dshoulder_elv', 'delbow_flexion', 'trial', 'subj']
# CALIB params
path_calib         = './data_CALIB'
name_expe          = 'CALIB'
save_path_calib    = './data_CALIB/dataMatCalib.pkl'
nb_subj            = 17
cond               = 'MJ'
nb_trials_calib    = 10
duration_idx_calib = 2900
params_calib       = {'path': path_calib, 'expeName': name_expe, 'save_path': save_path_calib, 'nb_subj': nb_subj, 'condition': cond,
                      'nb_trials': nb_trials_calib, 'dataTypes': data_type_list, 'listOfVariables': list_vars, 'durationIndex': duration_idx_calib}
# ASSIST params
path_assist         = './data_ASSIST'
name_expe           = 'ASSIST'
save_path_assist    = './data_ASSIST/dataMatAssist.pkl'
subjs_list          = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S09', 'S10', 'S12', 'S14']
conds_list          = ['T', 'ES', 'EG']
phases_list         = ['PE', 'PS', 'RE', 'RS']
nb_trials_calib     = 26
duration_idx_assist = 500
params_assist       = {'path': path_assist, 'expeName': name_expe, 'save_path': save_path_assist, 'subjList': subjs_list, 'conditions': conds_list,
                      'nb_trials': nb_trials_calib, 'phases': phases_list, 'dataTypes': data_type_list, 'listOfVariables': list_vars,
                      'durationIndex': duration_idx_assist}

In [4]:
## Get data matrices
# Import CALIB data
data_calib = getMat.get_data(src_data[0], params_calib)
print('CALIB data shape: ' + str(data_calib.shape))
# Import ASSIST data
data_assist = getMat.get_data(src_data[1], params_assist)
print('ASSIST data shape: ' + str(data_assist.shape))

Loaded calibration data from pickle.
CALIB data shape: (493000, 18)
Loaded assistance data from pickle.
ASSIST data shape: (227322, 18)


In [5]:
## Pre-processing parameters
savePath      = './all_params/folded_ablated_data/'
ablationsList = ['velocity', 'anthropo', 'forcestorques', 'forces', 'torques', 'Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
foldsList     = ['2', '5']
forceLoc      = 'wrist'

params_prepro = {'ablations': ablationsList, 'folds': foldsList, 'forceLoc': forceLoc, 'savePathPrepro': savePath}

In [6]:
## Prepare dictionnary with all ablations and foldings
fold_dict_all = mlu.get_all_folded_ablated_data(data_calib, data_assist, params_prepro)

Computing folded and ablated datasets...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments will be performed...
Number of subjects in CALIB is not divisible by number of folds. Adjustments 

In [11]:
## Mapping parameters
# Common
dataDir         = './all_params/folded_ablated_data/'
fittedModelsDir = './all_params/all_fitted_models/'
fitTypesList    = ['MVLR', 'MVPR'] # FNO, LSTM

# MVPR
degreesList = [2, 3] # Degree of the fitted polynomial for each input

# FNO
nbModesList   = [4, 8, 16, 32]       # Number of modes [number of low Fourier frequencies]
nbHChanList   = [8, 16, 32, 64] # Number of hidden channels [dimensionnality of the network's internal features]
nbMaxIterList = [100, 250, 500]      # Maximum number of training iterations
limLoss       = 1e-6                 # Convergence limit (if less improvement between two iterations then training stops)

# LSTM
# TODO

In [12]:
## Prepare lists of parameters dictionnaries to train different versions of the models
list_params_sets = []
for fitting in fitTypesList:
    for ablation in ablationsList:
        for fold in foldsList:
            fold_name = fold + '-fold'
            data_path = dataDir + ablation + '_' + fold_name + '.pkl'
            if fitting == 'MVLR':
                saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '.pkl'
                dict_params = {'model': fitting, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                               'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                list_params_sets.append(dict_params)
            elif fitting == 'MVPR':
                for degree in degreesList:
                    saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '_deg' + str(degree) + '.pkl'
                    dict_params = {'model': fitting, 'degree': degree, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                                   'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                    list_params_sets.append(dict_params)
            elif fitting == 'FNO':
                "FNO is not yet handled: Needs HPC"
            else:
                "The requested fitting type is not yet handled."

In [14]:
## Run training of models in parallel
# Create list to fill using manager
manager = Manager()
list_trainedModels = manager.list()
# Create multiprocessing pool
nb_processes = 3 #cpu_count() - 9
pool = Pool(processes = nb_processes)
# Run multiprocessed traning
out = pool.map(partial(trMap.run_training, list_trainedModels, list_params_sets), range(0, len(list_params_sets)))

Model already fitted and available at: ./all_params/all_fitted_models/MVLR_velocity_2-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_velocity_5-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_anthropo_2-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_anthropo_5-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_forcestorques_2-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_forcestorques_5-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_Ty_2-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_Ty_5-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_Tz_2-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_models/MVLR_Tz_5-fold.pkl
Model already fitted and available at: ./all_params/all_fitted_m